# BCOM — Rolling realized volatility

Computes and plots rolling 1-year (252-day) and 3-year (756-day) annualized realized volatility for the Bloomberg Commodity Index.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

from alpha_lab.data.loaders.yfinance import load_prices

TICKER = "^BCOM"          # Bloomberg Commodity Index; swap for COMB or PDBC if needed
START  = "2000-01-01"     # long start so 3Y rolling has room to warm up
WIN_1Y = 252
WIN_3Y = 756

## 1. Price history

In [ ]:
prices = load_prices(TICKER, start=START)
px = prices.squeeze()  # single-column → Series

fig, ax = plt.subplots(figsize=(12, 3.5), constrained_layout=True)
ax.plot(px.index, px.values, color="#2c7bb6", linewidth=1.2, label=TICKER)
ax.fill_between(px.index, px.values, px.min(), alpha=0.08, color="#2c7bb6")
ax.set_ylabel("Index level")
ax.set_title(f"{TICKER} — price history", fontsize=12, pad=8)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 2. Rolling realized volatility

Annualized realized vol = rolling std of log returns × √252.

In [ ]:
log_ret = np.log(px / px.shift(1)).dropna()

vol_1y = log_ret.rolling(WIN_1Y).std() * np.sqrt(252)
vol_3y = log_ret.rolling(WIN_3Y).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

ax.plot(vol_1y.index, vol_1y.values, color="#e74c3c", linewidth=1.2,
        label="1Y rolling vol")
ax.plot(vol_3y.index, vol_3y.values, color="#2c3e50", linewidth=1.5,
        linestyle="--", label="3Y rolling vol")

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("Annualized volatility")
ax.set_title(f"{TICKER} — rolling realized volatility", fontsize=12, pad=8)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 3. Current levels

In [ ]:
summary = pd.DataFrame({
    "1Y vol":  vol_1y,
    "3Y vol":  vol_3y,
}).dropna()

current = summary.iloc[[-1]].rename(index=lambda _: "Current")
stats   = summary.describe().loc[["mean", "50%", "min", "max"]].rename(index={"50%": "median"})

(
    pd.concat([current, stats])
    .style
    .format("{:.1%}")
    .set_caption(f"{TICKER} realized vol — current vs. history")
)